In [ ]:
from collections import defaultdict
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from gymnasium.wrappers import RecordVideo
import os

class QLearningAgent:
  def __init__(
      self,
      env: gym.Env,
      learn_rate: float = 0.1,
      initial_epsilon: float = 1.0,
      epsilon_decay: float = 0.995,
      min_epsilon: float = 0.01,
      discount_factor: float = 0.99,
  ):
    self.env = env
    self.q_values = defaultdict(lambda: np.zeros(env.action_space.n))

    self.lr = learn_rate
    self.discount_factor = discount_factor
    self.training_error = []

    self.epsilon = initial_epsilon
    self.final_epsilon = min_epsilon
    self.epsilon_decay = epsilon_decay

  def get_action(self, obs: tuple[int, int, bool], training = True) -> int:
    if training and np.random.random() < self.epsilon:
      return self.env.action_space.sample()
    else:
      return int(np.argmax(self.q_values[obs]))

  def update(self, obs: tuple[int, int, bool], action: int, reward: float, terminated: bool, next_obs: tuple[int, int, bool]):
    future_q_value = (not terminated) * np.max(self.q_values[next_obs])
    target = reward + self.discount_factor * future_q_value
    temporal_diff = target - self.q_values[obs][action]

    self.q_values[obs][action] += self.lr * temporal_diff
    self.training_error.append(temporal_diff)

  def decay_epsilon(self):
    self.epsilon = max(self.final_epsilon, self.epsilon * self.epsilon_decay)

def train_agent(n_episodes):
  env = gym.make("Blackjack-v1", sab=False)
  agent = QLearningAgent(env=env)
  episode_rewards = []

  for episode in range(n_episodes):
    obs, _ = env.reset()
    total_reward = 0
    done = False

    while not done:
      action = agent.get_action(obs)
      next_obs, reward, terminated, truncated, _ = env.step(action)
      done = terminated or truncated

      agent.update(obs, action, reward, done, next_obs)
      obs = next_obs
      total_reward += reward

    agent.decay_epsilon()
    episode_rewards.append(total_reward)

    if (episode + 1) % 1000 == 0:
      avg_reward = np.mean(episode_rewards[-1000:])
      print(f"Episode {episode + 1}/{n_episodes}, "
          f"Avg Reward (last 1000): {avg_reward:.2f}, "
          f"Epsilon: {agent.epsilon:.3f}")

  env.close()
  return agent, episode_rewards

def plot_results(episode_rewards):
    plt.figure(figsize=(10, 5))
    plt.plot(episode_rewards, alpha=0.3, label='Episode Reward')

    window = 500
    moving_avg = np.convolve(episode_rewards, np.ones(window)/window, mode='valid')
    plt.plot(range(window-1, len(episode_rewards)), moving_avg, label=f'{window}-Episode Moving Average')

    plt.xlabel('Episode')
    plt.ylabel('Total Reward')
    plt.title('Q-Learning Training Progress on Blackjack')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def test_agent(agent, n_episodes=10, record_video=False, video_folder='./videos'):
  if record_video:
    os.makedirs(video_folder, exist_ok=True)
    env = gym.make("Blackjack-v1", render_mode='rgb_array', sab=False)
    env = RecordVideo(env, video_folder, episode_trigger=lambda x: True, name_prefix='blackjack-learn')
  else:
    env = gym.make("Blackjack-v1", render_mode='human', sab=False)

  for episode in range(n_episodes):
    obs, _ = env.reset()
    total_reward = 0
    done = False

    while not done:
        action = agent.get_action(obs, training=False)
        obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        total_reward += reward

    print(f"Test Episode {episode + 1}: Reward = {total_reward}")

  env.close()

  if record_video:
      print(f"\nVideos saved to: {video_folder}")
      print(f"Look for files like: blackjack-learn-episode-*.mp4")

if __name__ == "__main__":
  print("Training Blackjack Q-agent...")
  agent, rewards = train_agent(n_episodes=100000)

  plot_results(rewards)

  print("\nTesting trained agent...")
  test_agent(agent, n_episodes=5, record_video=False)

  print("\nRecording video of trained agent...")
  test_agent(agent, n_episodes=3, record_video=True, video_folder='./blackjack_videos')